# 1. LLM Fundamentals and Architectures

## History and Evolution of Language Models

### From n-grams to Transformers

Language models have a rich history, evolving from simple statistical models to the complex neural networks we see today.

**1. Statistical Language Models (n-grams):**
These were among the first successful language models. An n-gram is a contiguous sequence of *n* items from a given sample of text or speech. The core idea is to predict the next word in a sequence based on the *n-1* preceding words. While effective for some tasks, they struggle with capturing long-range dependencies and handling unseen n-grams.

**2. Recurrent Neural Networks (RNNs):**
RNNs were a major breakthrough, introducing the concept of memory. They process sequences of data by maintaining a hidden state that captures information from previous steps. This allowed them to model dependencies over longer sequences than n-grams. However, they suffered from the vanishing gradient problem, making it hard to learn very long-range dependencies.

**3. Long Short-Term Memory (LSTM) Networks:**
LSTMs are a special kind of RNN, designed to overcome the vanishing gradient problem. They use a series of 'gates' (input, output, and forget gates) to regulate the flow of information, allowing them to remember information for much longer periods.

**4. The Transformer Architecture:**
Introduced in the paper "Attention Is All You Need" (2017), the Transformer architecture revolutionized NLP. It dispensed with recurrence and instead relied entirely on a mechanism called **self-attention**. This allows the model to weigh the importance of different words in the input sequence when processing a particular word, enabling it to capture complex relationships and long-range dependencies far more effectively than RNNs or LSTMs. This architecture is the foundation for most modern LLMs, including models like BERT and GPT.

In [ ]:
# Simple Bigram (n=2) Language Model Example
from collections import defaultdict, Counter
import re

# Sample text corpus
corpus = """
The quick brown fox jumps over the lazy dog. 
The lazy dog slept in the sun. 
The quick brown cat is not a dog.
"""

# Preprocess and tokenize the text
tokens = re.findall(r'\w+', corpus.lower()) + ['</s>'] # Add a stop symbol

# Create bigrams
bigrams = [(tokens[i], tokens[i+1]) for i in range(len(tokens)-1)]

# Build the bigram model (a simple frequency count)
bigram_model = defaultdict(Counter)
for w1, w2 in bigrams:
    bigram_model[w1][w2] += 1

def predict_next_word(word):
    word = word.lower()
    if word in bigram_model:
        # Return the most likely next word
        return bigram_model[word].most_common(1)[0][0]
    else:
        return "(unknown)"

# --- Test the model ---
current_word = "the"
next_word = predict_next_word(current_word)
print(f"After '{current_word}', the next word is likely to be: '{next_word}'")

current_word = "lazy"
next_word = predict_next_word(current_word)
print(f"After '{current_word}', the next word is likely to be: '{next_word}'")

current_word = "quick"
next_word = predict_next_word(current_word)
print(f"After '{current_word}', the next word is likely to be: '{next_word}'")

## Basic Anatomy of an LLM

### Tokens, Embeddings, Layers, and Parameters

To understand how LLMs work, we need to be familiar with their core components.

**1. Tokens:**
LLMs don't process raw text. Instead, they break text down into smaller units called **tokens**. A token can be a word, a part of a word (a subword), or a single character. This process, called **tokenization**, converts a text string into a sequence of integers, where each integer corresponds to a token in the model's vocabulary.

**2. Embeddings:**
These integer tokens are then mapped to high-dimensional vectors of real numbers called **embeddings**. These embeddings are not just random vectors; they are learned during the model's training process to capture the semantic meaning and context of the tokens. Words with similar meanings will have similar embedding vectors.

**3. Layers:**
LLMs are deep neural networks composed of multiple layers, typically stacked **Transformer blocks**. Each layer takes a sequence of embeddings as input and processes them to produce a new sequence of embeddings that incorporates more contextual information. The depth of the model (the number of layers) allows it to learn increasingly complex patterns and relationships in the data.

**4. Parameters:**
The **parameters** of an LLM are the weights and biases within its layers that are adjusted during the training process. These parameters are what store the 'knowledge' the model has learned from the training data. The number of parameters is a key measure of a model's size and capacity. Modern LLMs have billions or even trillions of parameters.

In [ ]:
# First, ensure you have the necessary libraries installed
!pip install transformers torch

from transformers import AutoTokenizer, AutoModel
import torch

# Load a pre-trained tokenizer and model (e.g., BERT)
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# --- 1. Tokenization ---
text = "Here is some text to encode."
inputs = tokenizer(text, return_tensors="pt") # "pt" stands for PyTorch tensors

print("--- Tokenization ---")
print(f"Original Text: {text}")
print(f"Token IDs: {inputs['input_ids']}")
print(f"Tokens: {tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])}")

# --- 2. Embeddings ---
# Get the embeddings for the input tokens
# We wrap this in a 'no_grad' context because we are not training the model
with torch.no_grad():
    outputs = model(**inputs)

# The embeddings are the last hidden states
embeddings = outputs.last_hidden_state

print("\n--- Embeddings ---")
print(f"Shape of the embeddings tensor: {embeddings.shape}")
print("(Batch Size, Sequence Length, Hidden Size)")

# --- 3. Parameters ---
num_params = model.num_parameters()
print("\n--- Parameters ---")
print(f"The model '{model_name}' has {num_params / 1e6:.2f} million parameters.")

## Transformer Architecture

### Self-Attention, Positional Encoding, and Feed-Forward Networks

The Transformer architecture, introduced in "Attention Is All You Need," is the foundation of most modern LLMs. It's built on a few key innovations:

**1. Self-Attention:**
This is the core mechanism of the Transformer. Instead of processing words in a fixed order, self-attention allows the model to look at all the other words in the input sequence and weigh their importance when processing a specific word. For each word, the model creates three vectors: a **Query** (Q), a **Key** (K), and a **Value** (V). The attention score is calculated by taking the dot product of the Query vector of the current word with the Key vectors of all other words. These scores are then scaled, passed through a softmax function to get weights, and used to create a weighted sum of the Value vectors. This result is a new representation of the word that is richly informed by its context.

**2. Positional Encoding:**
Since the self-attention mechanism doesn't inherently know the order of the words, Transformers need a way to incorporate positional information. This is done by adding **positional encodings** to the input embeddings. These are vectors that provide information about the position of each token in the sequence. The original paper used sine and cosine functions of different frequencies, but other methods exist.

**3. Multi-Head Attention:**
Instead of performing a single attention calculation, Transformers use **multi-head attention**. This means the model has multiple sets of Q, K, and V weight matrices, allowing it to jointly attend to information from different representation subspaces at different positions. It's like having multiple 'experts' looking at the sentence from different perspectives.

**4. Feed-Forward Networks (FFN):**
After the attention mechanism, the output for each position is passed through a simple, fully connected feed-forward network. This network is applied to each position separately and identically. It consists of two linear transformations with a ReLU activation in between, adding non-linearity and further processing capabilities to the model.

In [ ]:
# Ensure matplotlib is installed for visualization
!pip install matplotlib

import torch
from transformers import AutoTokenizer, AutoModel
import matplotlib.pyplot as plt
import numpy as np

# Load a model and tokenizer, configured to output attention weights
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name, output_attentions=True)

# --- Visualize Self-Attention ---
text = "The cat sat on the mat."
inputs = tokenizer(text, return_tensors='pt')
with torch.no_grad():
    outputs = model(**inputs)

# Attention weights have shape: (batch_size, num_heads, sequence_length, sequence_length)
attention = outputs.attentions[-1]  # Get the attention weights from the last layer
attention = attention[0, 0].detach().numpy() # Focus on the first head and batch item

tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

# Create a heatmap of the attention scores
fig, ax = plt.subplots(figsize=(6, 6))
im = ax.imshow(attention, cmap='viridis')

ax.set_xticks(np.arange(len(tokens)))
ax.set_yticks(np.arange(len(tokens)))
ax.set_xticklabels(tokens)
ax.set_yticklabels(tokens)

plt.setp(ax.get_xticklabels(), rotation=45, ha="right",
         rotation_mode="anchor")

ax.set_title("Self-Attention Heatmap (Last Layer, First Head)")
fig.tight_layout()
plt.show()

## Comparison: LLMs vs. Traditional NLP Models

### Comparison: LLMs vs. Traditional NLP Models

Here is a summary comparing modern Large Language Models (LLMs) with traditional NLP models.

| Feature | Traditional NLP Models (e.g., BoW, TF-IDF, Word2Vec, LSTMs) | Large Language Models (LLMs) (e.g., Transformers) |
| :--- | :--- | :--- |
| **Architecture** | Statistical, or recurrent neural networks. | Deep, attention-based neural networks (Transformers). |
| **Context Handling** | Limited or sequential context. Struggles with long-range dependencies. | Excellent at understanding deep, bidirectional, and long-range context via self-attention. |
| **Performance** | Good for specific, narrow tasks (e.g., classification, sentiment analysis). | State-of-the-art performance across a wide range of tasks (generation, summarization, Q&A). |
| **Pre-training** | Often trained on smaller, task-specific datasets. Word embeddings (like Word2Vec) were pre-trained, but not the full model. | Pre-trained on massive, diverse datasets, enabling strong zero-shot and few-shot learning. |
| **Flexibility** | Models are typically task-specific and require significant re-engineering for new tasks. | Highly flexible. The same pre-trained model can be fine-tuned for many different tasks with minimal changes. |
| **Key Strength** | Efficiency and simplicity for well-defined problems. | Generalization, contextual understanding, and text generation. |
| **Examples** | Naive Bayes, SVMs, TF-IDF, Word2Vec, GloVe, LSTMs, GRUs. | BERT, GPT series, T5, LLaMA, Claude. |

---

**Tip:** For each notebook, align practical code examples with key compliance and governance touchpoints—such as regulatory triggers, privacy protections during preprocessing, and documentation for auditability—to ensure relevance for enterprise and regulated sector use cases.